# D4 — Merge LoRA Adapter + Convert to GGUF for Ollama

Run this **after** `train_qlora_colab.ipynb` finishes. It:
1. Loads the base model in full precision.
2. Applies your LoRA adapter and merges it in.
3. Saves the merged full model to Drive.
4. Clones `llama.cpp` and converts the merged model to GGUF (Q4_K_M quantization).
5. Downloads the GGUF file to your laptop.

The resulting `.gguf` file is what Ollama needs.

## 1. Install + mount Drive

In [ ]:
!pip install -q -U \
    "transformers==4.44.2" \
    "peft==0.12.0" \
    "accelerate==0.33.0" \
    "sentencepiece" \
    "protobuf"

In [ ]:
!pip install -q -U "torchao>=0.16.0"
print("\n⚠️  Now restart: Runtime → Restart session")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.3 MB/s eta 0:00:00

⚠️  Now restart: Runtime → Restart session


In [ ]:
!pip install -q -U "tokenizers>=0.20" "transformers>=4.46"
print("\n⚠️  Now restart: Runtime → Restart session")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 49.2 MB/s eta 0:00:00

⚠️  Now restart: Runtime → Restart session


In [ ]:
!pip install -q -U "accelerate>=1.0.0"
print("\n⚠️  Restart: Runtime → Restart session")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 12.5 MB/s eta 0:00:00

⚠️  Restart: Runtime → Restart session


In [ ]:
!pip install -q -U "peft>=0.15.0" "torchao>=0.16.0"
print("\n⚠️  Restart: Runtime → Restart session")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 13.9 MB/s eta 0:00:00

⚠️  Restart: Runtime → Restart session


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ADAPTER_DIR = "/content/drive/MyDrive/d4_qlora_adapter/final"
MERGED_DIR  = "/content/drive/MyDrive/d4_qlora_merged"

import os
os.makedirs(MERGED_DIR, exist_ok=True)

assert os.path.exists(ADAPTER_DIR), f"Adapter not found at {ADAPTER_DIR}"
print("Adapter found.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter found.


## 2. Merge adapter into base model

Loaded in FP16 (not 4-bit) for clean merging.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("Loading base model in FP16...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,          # NEW name in transformers 4.46+
    device_map="auto",
    trust_remote_code=True,
)

print("Loading adapter on top...")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)

print("Merging adapter into base weights...")
model = model.merge_and_unload()

print("Saving merged model to Drive...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

print("Merged model saved to:", MERGED_DIR)
!ls -lh {MERGED_DIR}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading base model in FP16...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading adapter on top...
Merging adapter into base weights...
Saving merged model to Drive...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/drive/MyDrive/d4_qlora_merged
total 2.9G
-rw------- 1 root root 2.5K Jun  7 23:16 chat_template.jinja
-rw------- 1 root root 1.4K Jun  7 23:15 config.json
-rw------- 1 root root  242 Jun  7 23:15 generation_config.json
-rw------- 1 root root 2.9G Jun  7 23:16 model.safetensors
-rw------- 1 root root  694 Jun  7 23:16 tokenizer_config.json
-rw------- 1 root root  11M Jun  7 23:16 tokenizer.json


## 3. Set up llama.cpp for GGUF conversion

In [ ]:
%cd /content
!git clone https://github.com/ggerganov/llama.cpp.git
%cd llama.cpp
!pip install -q -r requirements.txt

/content
Cloning into 'llama.cpp'...
remote: Enumerating objects: 98582, done.
remote: Counting objects: 100% (349/349), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 98582 (delta 235), reused 151 (delta 151), pack-reused 98233 (from 4)
Receiving objects: 100% (98582/98582), 398.46 MiB | 17.14 MiB/s, done.
Resolving deltas: 100% (70408/70408), done.
/content/llama.cpp
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 98.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 29.7 MB/s eta 0:00:00
 

## 4. Convert HF model → GGUF (FP16) → quantize to Q4_K_M

Q4_K_M is the sweet spot: ~1GB file for a 1.5B model, runs fast on CPU, minimal quality loss.

In [ ]:
GGUF_FP16 = "/content/qwen-tuned-fp16.gguf"
GGUF_Q4   = "/content/drive/MyDrive/qwen-tuned-q4_k_m.gguf"

# Step 4a: HF → GGUF (FP16)
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outfile {GGUF_FP16} \
    --outtype f16

INFO:hf-to-gguf:Loading model: d4_qlora_merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {1536, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float16

In [ ]:
import json, shutil
from transformers import AutoTokenizer

cfg_path = f"{MERGED_DIR}/tokenizer_config.json"

# 1. Inspect what's in there
with open(cfg_path) as f:
    cfg = json.load(f)
print("extra_special_tokens:", cfg.get("extra_special_tokens"))

# 2. Remove the broken field (it's likely an empty list anyway)
cfg.pop("extra_special_tokens", None)

# 3. Also remove any other fields that could trip llama.cpp
for k in ["chat_template_kwargs"]:
    cfg.pop(k, None)

with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)
print("✅ Cleaned tokenizer_config.json")

# 4. Verify by loading the tokenizer
tok = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)
print(f"✅ Tokenizer loads cleanly: {tok.__class__.__name__}")

extra_special_tokens: ['<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']
✅ Cleaned tokenizer_config.json
✅ Tokenizer loads cleanly: Qwen2Tokenizer


In [ ]:
# Step 4b: Build llama-quantize tool then quantize
%cd /content/llama.cpp
!cmake -B build -DCMAKE_BUILD_TYPE=Release > /dev/null 2>&1
!cmake --build build --target llama-quantize -j4

!./build/bin/llama-quantize {GGUF_FP16} {GGUF_Q4} Q4_K_M

/content/llama.cpp
[  0%] Building CXX object common/CMakeFiles/llama-common-base.dir/build-info.cpp.o
[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
[  0%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
[  0%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml.cpp.o
[  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-alloc.c.o
[  2%] Linking CXX static library libllama-common-base.a
[  2%] Built target llama-common-base
[  4%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend.cpp.o
[  4%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-backend-meta.cpp.o
[  4%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-opt.cpp.o
[  4%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/ggml-threading.cpp.o
[  4%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml-quants.c.o
[  6%] Building CXX object ggml/src/CMakeFiles/ggml-base.dir/gguf.cpp.o
[  6%] Linking CXX shared libra

In [ ]:
import os
size_mb = os.path.getsize(GGUF_Q4) / 1e6
print(f"\n✅ Quantized GGUF: {GGUF_Q4}")
print(f"Size: {size_mb:.1f} MB")
print("\nThis file is now saved to your Google Drive.")
print("Download it to your local machine and put it in: models/qwen-tuned-q4_k_m.gguf")


✅ Quantized GGUF: /content/drive/MyDrive/qwen-tuned-q4_k_m.gguf
Size: 986.0 MB

This file is now saved to your Google Drive.
Download it to your local machine and put it in: models/qwen-tuned-q4_k_m.gguf


## 5. Download to your laptop

Two options:

**Option A — Direct download in Colab:**
```python
from google.colab import files
files.download(GGUF_Q4)
```

**Option B — Through Drive desktop app** (recommended for big files):
- File is already at `MyDrive/qwen-tuned-q4_k_m.gguf`
- Open Drive on your laptop, download it.

Once downloaded, move it to your project: `models/qwen-tuned-q4_k_m.gguf`

Then follow `docs/D4_TRAINING_GUIDE.md` step 4 to load it into Ollama.

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/qwen-tuned-q4_k_m.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>